In [ ]:
%pip install -r requirements-offline.txt --quiet

# Module M4 — สถิติที่ใช้ได้จริง

|  |  |
|---|---|
| **เวลา** | Day 2, 09:00–10:15 (75 นาที) · Offline (CPU) |
| **ข้อมูล** | Palmer Penguins · ราคาข้าวหอมมะลิ 105 (โบนัสไทย) |

**เมื่อจบโมดูลนี้ ผู้เรียนจะ...**

- อ่าน `describe()` ให้ออก และเลือกสถิติให้ตรงกับคำถาม
- เลือก mean หรือ median อย่างมีเหตุผล — รู้ว่าเมื่อไหร่ "ค่าเฉลี่ย" หลอกเรา
- วัดการกระจายด้วย std และ IQR แล้วแปลผลเป็นภาษาคน
- ดูรูปทรงข้อมูลด้วย histogram ก่อนเชื่อตัวเลขเดียว

**ข้อมูลที่ใช้ (citation):**

> Gorman, K. B., Williams, T. D. & Fraser, W. R. (2014). *Ecological
> sexual dimorphism and environmental variation within a common species
> biomass range: Antarctic penguins*. PLoS ONE 9(3):e90081 — ข้อมูล Palmer
> Penguins (CC0-1.0; package: Horst, Hill & Gorman 2020, doi:10.5281/zenodo.3960218)
>
> สำนักงานเศรษฐกิจการเกษตร (2565–2568). *ราคาข้าวเปลือกเจ้าหอมมะลิ 105
> รายเดือน* data.go.th (Open Data Common)

## 1. สถิติคือ "สรุปหลายพันตัวเลขให้เหลือไม่กี่ตัว"

ตารางเพนกวิน 344 แถว ตาเราอ่านไม่ไหว — สถิติเชิงพรรณนาคือการย่อ
ทั้งตารางให้เหลือตัวเลขไม่กี่ตัวที่บอกได้ว่า "ข้อมูลชุดนี้หน้าตาเป็นอย่างไร"

สองคำถามที่สถิติต้องตอบได้เสมอ:

1. **ค่าทั่วไปอยู่ตรงไหน** (ค่ากลาง — mean / median)
2. **ค่ากระจายกว้างแค่ไหน** (การกระจาย — std / IQR)

pandas คำนวณให้หมด — งานของเราไม่ใช่คำนวณ แต่คือ**เลือกให้ถูกคำถาม และแปลผลให้คนอ่านเข้าใจ**

In [ ]:
import pandas as pd
from pathlib import Path


def repo_root() -> Path:
    """หา repo root โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "requirements-offline.txt").exists():
            return p
    raise SystemExit(
        "ไม่พบ repo root — เปิด JupyterLab จาก repo root (uv run jupyter lab)")

DATA = repo_root() / "datasets" / "palmer_penguins"  # noqa: E305

penguins = pd.read_csv(DATA / "penguins.csv")
# ค่าหาย 19 ช่องจาก 344 แถว (สอนใน M3) — แล็บนี้ตัดทิ้งเพื่อความเรียบง่าย
# (bare .dropna() = ตัดแถวที่มีช่องว่าง "ที่ไหนก็ได้" ทิ้งทั้งแถว)
penguins = penguins.dropna()
print(f"เหลือ {len(penguins)} แถวจาก 344")

## 2. ค่ากลาง — เริ่มจากคำสั่งเดียว: `.describe()`

คำสั่งเดียวที่ควรพิมพ์ทุกครั้งก่อนแตะตารางตัวเลข:

In [ ]:
penguins["body_mass_g"].describe()

อ่านแต่ละบรรทัดให้ออก:

| แถว | คืออะไร |
|---|---|
| `count` | จำนวนค่าที่ไม่ว่าง — ดูก่อนว่าตัวเลขคิดจากกี่แถว |
| `mean` | ค่าเฉลี่ย (ผลรวม ÷ จำนวน) |
| `std` | ความกระจายมาตรฐาน — ดู §3 |
| `min` / `max` | ค่าต่ำสุด / สูงสุด — จุดตรวจ "ค่าแปลกหรือเปล่า" |
| `25%` / `50%` / `75%` | ควอร์ไทล์ — 50% คือ median ตัวจริง |

**โน้ตเก่าที่ต้องแก้:** คนมักจำกันว่า "เฉลี่ย = ตัวแทนข้อมูล" — จริงบางทีเท่านั้น

In [ ]:
# ทำไม? ลองกับตัวเลขเงินเดือนทีมงาน 7 คน (หน่วย: บาท)
salaries = [25_000, 28_000, 30_000, 32_000, 35_000, 40_000, 250_000]
sal = pd.Series(salaries)   # แปลง list เป็น Series ใช้เครื่องมือ pandas ได้
print(f"mean   = {sal.mean():,.0f}")
print(f"median = {sal.median():,.0f}")

mean **62,857** บาท — แต่ทีมนี้ไม่มีใครได้เท่านั้น! เงินเดือนผู้บริหารคนเดียว
ลากค่าเฉลี่ยลอยขึ้นไป ส่วน median (ตัวกลางตัวจริง = 32,000) ยังบอกเรื่อง
"คนทั่วไปในทีม" ได้

**กฎนิ้วโป้ง:** ข้อมูลเรื่อง**เงิน ราคา เวลารอ** มักเบ้ (มีค่าสูงผิดปกติไม่กี่ตัว) —
รายงานด้วย **median** ไว้ก่อน แล้วเสริม mean ถ้าจำเป็น

## 3. การกระจาย — ค่ากลางเท่ากันอาจมาจากคนละโลก

สองทีมซ่อมเครื่อง ใช้เวลาเฉลี่ยเท่ากัน 30 นาที/เครื่อง:

- ทีม A: ทุกงานอยู่ช่วง 28–32 นาที — นิ่ง โปรแกรมงานได้
- ทีม B: บางงาน 10 นาที บางงาน 90 นาที — เฉลี่ยเท่ากัน แต่ลูกค้าโทรด่าต่างกัน

สถิติที่จับเรื่องนี้ได้สองตัว:

In [ ]:
mass = penguins["body_mass_g"]

print(f"std  = {mass.std():,.0f} g   (ระยะเฉลี่ยที่ค่าหลุดจาก mean)")

q1, q3 = mass.quantile([0.25, 0.75])
print(f"IQR  = {q3 - q1:,.0f} g   (กลาง 50% ของข้อมูลอยู่ช่วง "
      f"{q1:,.0f}–{q3:,.0f} g)")

- **std** — ตัวเลขยิ่งใหญ่ = ข้อมูลยิ่งกระจัดกระจาย หน่วยเดียวกับข้อมูล (กรัม)
- **IQR** (ช่วง Q3 − Q1) — ความกว้างของ "กลาง 50%" ไม่สนใจค่าสุดขอบสองข้าง
  จึงไม่โดน outlier ลาก

**ตอนรายงาน:** ตัวเลขเดี่ยวไม่มีวันพอ — "เฉลี่ย 4,200 กรัม" ต้องแถม
"± std 800" หรือ "กลาง 50% อยู่ 3,550–4,775" ถึงจะเชื่อได้

## 4. สรุปต่อกลุ่ม — สถิติมีประโยชน์จริงเมื่อเอามา**เทียบกัน**

`.groupby()` + `.agg()` เรียนแล้วใน M2 — คราวนี้ขอเพิ่ม `median`/`std`
เข้าชุดการรวม แล้วอ่านตารางให้เป็นเรื่อง:

In [ ]:
by_species = penguins.groupby("species").agg(
    n=("body_mass_g", "size"),
    mean_g=("body_mass_g", "mean"),
    median_g=("body_mass_g", "median"),
    std_g=("body_mass_g", "std"),
)
by_species.round(0)

อ่านออกมาเป็นภาษาคน:

- **Gentoo** หนักกว่าสองชนิดที่เหลือชัดเจน (~5,000 ก. vs ~3,700 ก.)
- **Adelie vs Chinstrap** — mean ใกล้กันมาก แต่ก่อนตัดสินใจให้ดู std และ n
- คอลัมน์ `n` สำคัญเสมอ — กลุ่มที่มีข้อมูล 10 แถวเทียบกับ 300 แถวไม่ได้เลย

## 5. มองรูปทรงด้วย histogram — ก่อนเชื่อ "ค่าเฉลี่ยรวม"

ตัวเลขใน §2–§4 สรุปทั้ง 333 ตัวรวมกัน — มาดูว่าของจริงกระจายหน้าตาอย่างไร:

In [ ]:
import matplotlib.pyplot as plt  # noqa: E402

plt.rcParams["font.family"] = [
    "DejaVu Sans", "Noto Sans Thai", "Leelawadee UI", "Tahoma",
]

fig, ax = plt.subplots(figsize=(7, 3))
penguins["body_mass_g"].plot.hist(bins=30, ax=ax, color="#457b9d")
ax.set_title("น้ำหนักเพนกวินทั้ง 333 ตัว")
ax.set_xlabel("น้ำหนัก (กรัม)")
plt.show()

กราฟไม่เป็นภูเขาลูกเดียว — มี**สองยอด** (bimodal)

ค่าเฉลี่ยรวม ~4,200 ก. ตกอยู่**ร่องกลาง**ระหว่างสองยอด — แปลว่า
"เพนกวินเฉลี่ยหนัก 4,200 ก." ไม่ตรงกับเพนกวินตัวไหนเลย!

เดาก่อนว่าอะไรแบ่งสองยอด — แล้วลองตรวจ:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for sex_name, color in [("female", "#e07a5f"), ("male", "#457b9d")]:
    mask = penguins["sex"] == sex_name
    penguins.loc[mask, "body_mass_g"].plot.hist(
        bins=25, alpha=0.55, ax=ax, color=color, label=sex_name)
ax.legend()
ax.set_title("น้ำหนักแยกตามเพศ — ต้นตอของสองยอด")
ax.set_xlabel("น้ำหนัก (กรัม)")
plt.show()

สองยอดมาจาก**เพศ** — ตัวผู้หนักกว่าตัวเมียชัดเจนในทุกชนิด

`.loc[mask, "body_mass_g"]` คือการเลือก "แถวตามเงื่อนไข + คอลัมน์ที่ต้องการ"
พร้อมกันในคำสั่งเดียว (ประโยคเดียวแทนสองขั้น `df[mask]["col"]`)

**บทเรียนหัวใจของโมดูลนี้:** เห็น mean เดียวอย่างเพิ่งเชื่อ — วาด histogram
ดูรูปทรงก่อนเสมอ ถ้ารูปทรงคล้ายกระดิ่งลูกเดียว mean+std พอ
ถ้าหลายยอด = มีกลุ่มซ่อนอยู่ ต้องแยกด้วย `groupby` ก่อนพูดอะไรต่อ

## 6. โบนัสไทย — ราคาข้าวหอมมะลิ: ข้อมูลเงินใช้ median

ราคาข้าวเปลือกหอมมะลิ 105 รายเดือน (สำนักงานเศรษฐกิจการเกษตร, บาท/ตัน)
โจทย์คลาสสิกของข้อมูลไทย: **ปีเป็น พ.ศ.** (2565 = ค.ศ. 2022)

In [ ]:
rice = pd.read_csv(
    repo_root() / "datasets" / "rice_price_105" / "rice105_monthly.csv")
print(f"{len(rice)} เดือน · ปี พ.ศ. {rice['year'].min()}"
      f"–{rice['year'].max()}")
rice["Value"].describe().round(0)

In [ ]:
# เทียบต่อปี: mean กับ median ต่างกันแค่ไหน
rice.groupby("year").agg(
    median_price=("Value", "median"),
    mean_price=("Value", "mean"),
    max_price=("Value", "max"),
).round(0)

อ่านตาราง: median ต่อปีไต่ขึ้นเรื่อย ๆ (13,184 → 15,288 บาท/ตัน) — และ
mean กับ median ใกล้กันมาก ข้อมูลชุดนี้เบ้ไม่มาก

จุดสอนสำคัญ: กฎ "ข้อมูลเงินใช้ median" เป็นกฎนิ้วโป้ง **ไม่ใช่กฎเหล็ก** —
เลือกตัวเลขไหน ต้องดูข้อมูลจริงก่อนเสมอ (หมายเหตุ: ปี 2568 มีเพียง 8 เดือน
ม.ค.–ส.ค. — อย่าเทียบกับปีเต็มโดยตรง)

## 7. เช็คลิสต์ "สถิติที่ใช้ได้จริง" + สรุป

ก่อนส่งตัวเลขให้ใครอ่าน ตรวจ 5 ข้อ:

1. **พิมพ์ `.describe()` ก่อนพูดอะไร** — count/mean/std/min/quartiles/max
2. ข้อมูลเงิน/ราคา/เวลารอ → **median** เป็นตัวเลขหลัก
3. ตัวเลขเดี่ยวไม่มีวันพอ — แนบ **std หรือ IQR** และ **หน่วย** ไปด้วย
4. เฉลี่ยรวมดูน่าสงสัย → **histogram ดูรูปทรง** ก่อน (bimodal = กลุ่มซ่อนอยู่)
5. เทียบกลุ่ม → `groupby().agg()` และ **ห้ามลืมคอลัมน์ n**

**ต่อไป (M5):** ตัวเลขพวกนี้จะกลายเป็นภาพ — กราฟที่สื่อสารได้
เพราะผู้อ่านรายงานจริง ๆ มองกราฟก่อนตารางเสมอ